In [ ]:
from datasets import load_dataset
ds = load_dataset("GEM/FairytaleQA")

In [ ]:
ds

In [ ]:
columns_to_keep = ['gem_id', 'content']
columns_to_remove = [col for col in ds['train'].column_names if col not in columns_to_keep]

In [ ]:
transformed_dataset = ds.remove_columns(columns_to_remove)
transformed_dataset = transformed_dataset.rename_column('gem_id', 'id')
transformed_dataset = transformed_dataset.rename_column('content', 'text')

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
transformed_dataset["dev"] = transformed_dataset.pop("validation")

In [ ]:
transformed_dataset

In [ ]:
transformed_dataset.push_to_hub("brimmann2/fairytaleqa1")

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from torchview import draw_graph
import os

# --- This part is the same and works correctly ---

# Ensure the output directory exists
os.makedirs("model_visualizations", exist_ok=True)

# 1. Load the model and tokenizer
model_name = "google/gemma-3-1b-it" # Using your new model
print(f"Loading model: {model_name}")
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("Model loaded successfully.")
model.eval()

# 2. Create a dummy input
dummy_input = tokenizer(
    "This is a sample input for visualization.",
    return_tensors="pt"
)

# 3. Generate the high-level torchview graph (This is your best bet for the paper)
print("Generating high-level overview visualization with torchview...")
try:
    model_graph_overview = draw_graph(
        model,
        input_data=dummy_input['input_ids'],
        graph_name='Gemma-3.1B High-Level Architecture',
        save_graph=True,
        filename=os.path.join("model_visualizations", "gemma_3.1b_overview"),
        roll=True,
        depth=1,
    )
    print("High-level visualization saved to model_visualizations/gemma_3.1b_overview.pdf")
except Exception as e:
    print(f"Failed to generate high-level torchview graph. Error: {e}")


# --- New Section: Using torchviz for a detailed graph ---

from torchviz import make_dot

print("\nGenerating detailed computation graph with torchviz...")
try:
    # 1. Perform a forward pass to trace the operations
    dummy_input_ids = dummy_input['input_ids']
    outputs = model(dummy_input_ids)
    logits = outputs.logits

    # 2. Generate the graph from the output tensor
    # The graph is traced backwards from this output.
    viz_graph = make_dot(logits, params=dict(model.named_parameters()), show_attrs=False, show_saved=False)
    viz_graph.format = 'pdf'
    viz_graph.render(os.path.join("model_visualizations", "gemma_3.1b_torchviz_detailed"), cleanup=True)
    print("Detailed torchviz graph saved to model_visualizations/gemma_3.1b_torchviz_detailed.pdf")

except ImportError:
    print("Please install torchviz (`pip install torchviz`) and Graphviz to generate the detailed graph.")
except Exception as e:
    print(f"Failed to generate torchviz graph. Error: {e}")


Loading model: google/gemma-3-1b-it
Model loaded successfully.
Generating high-level overview visualization with torchview...
High-level visualization saved to model_visualizations/gemma_3.1b_overview.pdf

Generating detailed computation graph with torchviz...
Failed to generate torchviz graph. Error: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
